# Fix merged dataset constraints — v7.4 domain-specific + manual JSON + Kinopoisk L5 fix

Этот ноутбук пересобирает `constraints` в финальном merged JSONL по доменным правилам, а не одним общим эвристическим парсером.

Ключевая схема:
- `constraints` — только чистые человекочитаемые условия, которые реально составляют запрос.
- `full_constraints` — машинная/аудитная информация: SPARQL evidence, старые/new hashes, quoted values.
- служебные поля (`source_dataset`, `template_id`, `source_file`, QID, `text_clues`) не попадают в clean `constraints`.

Ноутбук рассчитан на запуск из корня репозитория.

## Что исправляет v7.4

После просмотра генераторного ноутбука стало понятно, что constraints нельзя чинить универсальным `property -> label` алгоритмом: у `cinema`, `countries`, `dishes`, `smartphones`, `geo`, `paintings` и других доменов разные шаблоны.

Эта версия делает доменные extractors:
- `cinema`: genre, country, year, rating source (`rating_kinopoisk_min`, `rating_imdb_min`), franchise/series negation, actor/director, length.
- `countries`: region, official language, currency, population range/min/max, current/historical, EU/G7 membership/negative membership.
- `dishes`: include/exclude ingredients, cuisine, country, category.
- остальные домены: books, videogames, music_albums, software, people/scientists/mathematicians, paintings, museums, spacecraft, smartphones, universities, airports, cars, geo/geo_ru.

Дополнительно v7.4.3 создаёт JSON-файл для ручной валидации: список объектов с двумя полями `запрос` и `констрейнты`. CSV/XLSX не создаются.

In [1]:

from pathlib import Path
import json, re, csv, hashlib
from copy import deepcopy
from typing import Any, Dict, List, Optional, Tuple

FILENAME = "multihop_benchmark_merged_wikidata_format.jsonl"
CANDIDATE_INPUTS = [
    Path("merge_slices/merged_dataset") / FILENAME,
    Path("merge slices/merged_dataset") / FILENAME,
    Path("merge_slices/merged dataset") / FILENAME,
    Path("merge slices/merged dataset") / FILENAME,
]
INPUT_JSONL = next((p for p in CANDIDATE_INPUTS if p.exists()), CANDIDATE_INPUTS[0])
OUTPUT_JSONL = INPUT_JSONL.with_name(INPUT_JSONL.stem + ".clean_constraints_v7_4_domain_specific" + INPUT_JSONL.suffix)
DIFF_CSV = INPUT_JSONL.with_name("constraints_clean_v7_4_diff.csv")
AUDIT_CSV = INPUT_JSONL.with_name("constraints_clean_v7_4_audit.csv")
OVERWRITE_INPUT = False

# Keys that must never appear in clean constraints.
FORBIDDEN_CLEAN_KEYS = {
    "source_dataset", "source", "source_file", "template_id", "template_family",
    "is_advanced", "gold_truncated", "created_at", "id", "domain", "complexity",
    "sparql_query", "ask_validator_sparql", "gold_answer_qids", "gold_answer_labels_ru",
    "method", "warnings", "text_clues", "full_constraints", "constraints_fix_meta",
}

KIND_BY_DOMAIN = {
    "cinema": "film",
    "books": "book",
    "videogames": "video_game",
    "music_albums": "music_album",
    "software": "software",
    "people": "person",
    "scientists": "person",
    "mathematicians": "person",
    "paintings": "painting",
    "museums": "museum",
    "spacecraft": "spacecraft",
    "countries": "country",
    "dishes": "dish",
    "smartphones": "smartphone",
    "universities": "university",
    "airports": "airport",
    "cars": "car_model",
    "geo": "geo_object",
    "geo_ru": "geo_object",
}

def stable_hash(obj: Any) -> str:
    return hashlib.md5(json.dumps(obj, ensure_ascii=False, sort_keys=True).encode("utf-8")).hexdigest()

def norm_space(s: Any) -> str:
    if s is None:
        return ""
    s = str(s).replace("\u00a0", " ").replace("\u202f", " ")
    return re.sub(r"\s+", " ", s).strip()

def clean_label(s: Any) -> Optional[str]:
    s = norm_space(s).strip(" ,.;:«»\"'")
    return s or None

def qtext(rec: Dict[str, Any]) -> str:
    return norm_space(rec.get("query_text_ru") or "")

def quoted_values(text: str) -> List[str]:
    vals = re.findall(r"[«\"]([^»\"]+)[»\"]", text)
    return [clean_label(v) for v in vals if clean_label(v)]

def parse_int_ru(s: str) -> int:
    return int(re.sub(r"[^\d-]", "", s))

def add_if(d: Dict[str, Any], key: str, value: Any) -> None:
    if value is None:
        return
    if isinstance(value, str):
        value = clean_label(value)
        if not value:
            return
    if isinstance(value, list):
        value = [x for x in value if x not in (None, "", [])]
        if not value:
            return
    d[key] = value

def add_kind(clean: Dict[str, Any], domain: str) -> None:
    k = KIND_BY_DOMAIN.get(domain)
    if k:
        clean["kind"] = k

def year_constraints(text: str) -> Dict[str, int]:
    t = norm_space(text)
    out = {}
    # explicit range: 2018–2023 / 2018-2023
    m = re.search(r"(?<!\d)(1[5-9]\d{2}|20\d{2}|21\d{2})\s*[–—-]\s*(1[5-9]\d{2}|20\d{2}|21\d{2})(?!\d)", t)
    if m:
        y1, y2 = int(m.group(1)), int(m.group(2))
        out["year_from"], out["year_to"] = min(y1, y2), max(y1, y2)
        return out
    # "2015 года"
    m = re.search(r"(?<!\d)(1[5-9]\d{2}|20\d{2}|21\d{2})(?!\d)\s*(?:года|год|г\.)", t, flags=re.I)
    if m:
        y = int(m.group(1)); out["year_from"] = y; out["year_to"] = y
        return out
    # thresholds
    m = re.search(r"(?:после|позже|с|начиная с)\s+(1[5-9]\d{2}|20\d{2}|21\d{2})", t, flags=re.I)
    if m:
        out["year_from"] = int(m.group(1))
    m = re.search(r"(?:до|раньше|не позже|по)\s+(1[5-9]\d{2}|20\d{2}|21\d{2})", t, flags=re.I)
    if m:
        out["year_to"] = int(m.group(1))
    return out

def numeric_range_after(text: str, anchors: List[str]) -> Tuple[Optional[int], Optional[int]]:
    t = norm_space(text)
    anchor_alt = "|".join(re.escape(a) for a in anchors)
    m = re.search(rf"(?:{anchor_alt}).{{0,80}}?диапазон[ае]?\s+([\d\s]+)\s*[–—-]\s*([\d\s]+)", t, flags=re.I)
    if not m:
        m = re.search(rf"(?:{anchor_alt}).{{0,80}}?([\d\s]{{2,}})\s*[–—-]\s*([\d\s]{{2,}})", t, flags=re.I)
    if m:
        return parse_int_ru(m.group(1)), parse_int_ru(m.group(2))
    return None, None

def _safe_float(raw: str) -> Optional[float]:
    """Parse a numeric capture safely. Returns None for empty/space-only captures."""
    if raw is None:
        return None
    s = re.sub(r"\s+", "", str(raw)).replace(",", ".")
    if not re.search(r"\d", s):
        return None
    try:
        return float(s)
    except ValueError:
        return None

def numeric_min_after(text: str, anchors: List[str]) -> Optional[float]:
    t = norm_space(text)
    anchor_alt = "|".join(re.escape(a) for a in anchors)
    pats = [
        rf"(?:{anchor_alt}).{{0,80}}?(?:не ниже|не менее|минимум|от|больше|выше|>=)\s*([0-9][0-9\s]*(?:[.,][0-9]+)?)",
        rf"(?:{anchor_alt}).{{0,80}}?(?:>=)\s*([0-9][0-9\s]*(?:[.,][0-9]+)?)",
    ]
    for p in pats:
        m = re.search(p, t, flags=re.I)
        if m:
            val = _safe_float(m.group(1))
            if val is not None:
                return val
    return None

def numeric_max_after(text: str, anchors: List[str]) -> Optional[float]:
    t = norm_space(text)
    anchor_alt = "|".join(re.escape(a) for a in anchors)
    for p in [
        rf"(?:{anchor_alt}).{{0,80}}?(?:не выше|не более|максимум|до|меньше|ниже|<=)\s*([0-9][0-9\s]*(?:[.,][0-9]+)?)",
    ]:
        m = re.search(p, t, flags=re.I)
        if m:
            val = _safe_float(m.group(1))
            if val is not None:
                return val
    return None

def first_quoted_after(text: str, anchors: List[str]) -> Optional[str]:
    t = norm_space(text)
    anchor_alt = "|".join(re.escape(a) for a in anchors)
    m = re.search(rf"(?:{anchor_alt})[^«\"]*[«\"]([^»\"]+)[»\"]", t, flags=re.I)
    return clean_label(m.group(1)) if m else None

def text_after_marker(text: str, patterns: List[str], stop=r"[,.;]") -> Optional[str]:
    t = norm_space(text)
    for p in patterns:
        m = re.search(p, t, flags=re.I)
        if m:
            return clean_label(m.group(1))
    return None

def country_from_text(text: str) -> Optional[str]:
    patterns = [
        r"из страны:\s*([^,.;]+)",
        r"страны:\s*([^,.;]+)",
        r"страна\s*[—\-:]\s*([^,.;]+)",
        r"из страны\s+[«\"]?([^,»\".;]+)[»\"]?",
        r"странах?\s+[«\"]?([^,»\".;]+)[»\"]?",
    ]
    return text_after_marker(text, patterns)

def extract_rating(text: str, clean: Dict[str, Any]) -> None:
    t = norm_space(text)
    # Kinopoisk / Кинопоиск
    for src_key, names in [
        ("kinopoisk", ["Кинопоиск", "КиноПоиск", "KP", "kp"]),
        ("imdb", ["IMDb", "IMDB", "imdb"]),
    ]:
        source_pat = "|".join(re.escape(x) for x in names)
        # "рейтинг по Кинопоиск не ниже 7"
        m = re.search(rf"рейтинг\w*(?:\s+по)?\s+(?:{source_pat}).{{0,40}}?(не ниже|не менее|от|выше|больше|>=)\s*([0-9]+(?:[.,][0-9]+)?)", t, flags=re.I)
        if m:
            clean[f"rating_{src_key}_min"] = float(m.group(2).replace(",", "."))
        m = re.search(rf"рейтинг\w*(?:\s+по)?\s+(?:{source_pat}).{{0,40}}?(не выше|не более|до|меньше|<=|(?<!не\s)ниже)\s*([0-9]+(?:[.,][0-9]+)?)", t, flags=re.I)
        if m:
            clean[f"rating_{src_key}_max"] = float(m.group(2).replace(",", "."))
    # "Rating Кинопоиск >= 7" inside fake sparql/comment also supported through rec text only if present.

def extract_franchise_flags(text: str, clean: Dict[str, Any]) -> None:
    t = norm_space(text).lower()
    if re.search(r"(не\s+явля[ею]тся|не\s+является|не\s+часть|не\s+вход[яи]т).{0,60}(франшиз|сер(ии|ия)|киносер)", t):
        clean["not_franchise"] = True
    elif re.search(r"(частью|часть|вход[яи]т).{0,40}(франшиз|сер(ии|ия)|киносер)", t):
        clean["part_of_franchise"] = True
    if re.search(r"не\s+(?:сиквел|продолжение)", t):
        clean["not_sequel"] = True
    if re.search(r"не\s+ремейк", t):
        clean["not_remake"] = True

def clean_from_old_safe(old: Dict[str, Any], clean: Dict[str, Any], key_map: Dict[str, str]) -> None:
    """Use only human-readable old keys, never qids/meta, and only if the key is not already detected."""
    if not isinstance(old, dict):
        return
    for old_key, new_key in key_map.items():
        if new_key in clean:
            continue
        val = old.get(old_key)
        if val is None or val == "" or str(val).startswith("Q"):
            continue
        if old_key.endswith("_qid") or old_key.endswith("_qids"):
            continue
        add_if(clean, new_key, val)

def parse_sparql_machine(rec: Dict[str, Any]) -> Dict[str, Any]:
    """Machine evidence only; not used as clean labels by itself."""
    sparql = (rec.get("sparql_query") or "") + "\n" + (rec.get("ask_validator_sparql") or "")
    prop_hits = []
    for m in re.finditer(r"wdt:(P\d+)(?:/wdt:P\d+\*)?\s+wd:(Q\d+)", sparql):
        prop_hits.append({"property_id": m.group(1), "qid": m.group(2)})
    filters = re.findall(r"FILTER\s*\((.*?)\)", sparql, flags=re.I | re.S)
    return {"property_qid_hits": prop_hits, "filters": [norm_space(f) for f in filters]}

def base_clean(rec: Dict[str, Any]) -> Tuple[Dict[str, Any], Dict[str, Any], List[str]]:
    domain = rec.get("domain")
    clean = {}
    add_kind(clean, domain)
    full = {"method": "domain_specific_constraints_v7_4", "machine_evidence": parse_sparql_machine(rec)}
    warnings = []
    return clean, full, warnings

# ---------------- Domain cleaners ----------------



def minutes_from_length_phrase(phrase: str) -> Tuple[Optional[int], Optional[int]]:
    """Return (min_minutes, max_minutes) from Russian movie length phrase.
    Handles: не более 3 часов, не более 2 часов, более 2 часов, 1.5 часов / 90 минут.
    """
    p = norm_space(phrase).lower().replace("ё", "е")
    def to_minutes(num_raw: str, unit_hint: str) -> Optional[int]:
        val = _safe_float(num_raw)
        if val is None:
            return None
        if "час" in unit_hint:
            return int(round(val * 60))
        return int(round(val))
    # explicit minutes in parentheses has priority: "(90 минут)"
    m_minute = re.search(r"([0-9][0-9\s]*(?:[.,][0-9]+)?)\s*(?:минут|мин\b)", p, flags=re.I)
    m_hour = re.search(r"([0-9][0-9\s]*(?:[.,][0-9]+)?)\s*(?:час|часа|часов)", p, flags=re.I)
    if m_minute:
        value = to_minutes(m_minute.group(1), "минут")
    elif m_hour:
        value = to_minutes(m_hour.group(1), "час")
    else:
        value = None
    if value is None:
        return None, None
    # Negated upper-bound patterns must be checked before "более".
    if re.search(r"не\s+более|не\s+дольше|до\s+[0-9]|максимум", p):
        return None, value
    if re.search(r"\bболее\b|дольше|свыше", p):
        return value, None
    return None, value


def apply_kinopoisk_cinema_old_constraints(rec: Dict[str, Any], clean: Dict[str, Any]) -> None:
    """For generated Kinopoisk L5 examples, the generator stored reliable structured constraints.
    Normalize them into clean human constraints and drop meta keys like type/source_dataset.
    """
    old = rec.get("constraints") or {}
    if not isinstance(old, dict):
        return
    is_kp_l5 = (
        rec.get("template_family") == "kinopoisk_api"
        or rec.get("template_id") == "cinema_l5_length"
        or str(rec.get("id", "")).startswith("cinema_l5_length")
        or old.get("type") == "cinema_l5_length"
    )
    if not is_kp_l5:
        return
    if old.get("genre_ru") and "genre_ru" not in clean:
        clean["genre_ru"] = old.get("genre_ru")
    if old.get("actor_ru") and "actor_ru" not in clean:
        clean["actor_ru"] = old.get("actor_ru")
    rng = old.get("movie_length_range")
    if isinstance(rng, (list, tuple)) and len(rng) >= 2:
        mn, mx = rng[0], rng[1]
        try:
            mn_i, mx_i = int(mn), int(mx)
            # Store only meaningful side(s); 1-180 means "not more than 3 hours".
            if mn_i > 1:
                clean["movie_length_min_minutes"] = mn_i
            if mx_i < 1000:
                clean["movie_length_max_minutes"] = mx_i
        except Exception:
            pass
    rating = old.get("rating")
    if isinstance(rating, (list, tuple)) and len(rating) >= 3:
        src, op, val = rating[0], rating[1], rating[2]
        src_l = str(src).lower()
        if src_l in {"kp", "kinopoisk", "кинопоиск"}:
            src_key = "kinopoisk"
        elif src_l == "imdb":
            src_key = "imdb"
        else:
            src_key = re.sub(r"\W+", "_", src_l).strip("_") or "unknown"
        try:
            val_f = float(val)
            if str(op) in {">=", ">", "min", "от", "не ниже"}:
                clean[f"rating_{src_key}_min"] = val_f
            elif str(op) in {"<=", "<", "max", "до", "не выше"}:
                clean[f"rating_{src_key}_max"] = val_f
        except Exception:
            pass
def clean_cinema(rec):
    clean, full, warnings = base_clean(rec)
    text = qtext(rec)
    old = rec.get("constraints") or {}

    # Text-first fields.
    add_if(clean, "genre_ru", first_quoted_after(text, ["жанра", "жанре", "в жанре"]))
    add_if(clean, "country_ru", country_from_text(text))
    clean.update(year_constraints(text))
    extract_rating(text, clean)
    extract_franchise_flags(text, clean)

    # Generated Kinopoisk L5 examples have reliable structured constraints in the old field.
    # Use them, but normalize into clean human-readable keys only.
    apply_kinopoisk_cinema_old_constraints(rec, clean)

    # actor/director bridge from natural text
    m = re.search(r"с участием акт[её]ра\s+([^,.;]+?)(?:\s+которые|\s+который|\s+и\s+|[,.;]|$)", text, flags=re.I)
    if m: add_if(clean, "actor_ru", m.group(1))
    m = re.search(r"с участием актрисы\s+([^,.;]+?)(?:\s+которые|\s+которая|\s+и\s+|[,.;]|$)", text, flags=re.I)
    if m: add_if(clean, "actor_ru", m.group(1))
    if re.search(r"режисс[её]р(?:ом)?\s*[-—]?\s*женщин", text, flags=re.I) or re.search(r"женщин[аы].{0,20}режисс", text, flags=re.I):
        clean["female_director"] = True
    if re.search(r"режисс[её]р(?:ом)?\s*[-—]?\s*мужчин", text, flags=re.I) or re.search(r"мужчин[аы].{0,20}режисс", text, flags=re.I):
        clean["male_director"] = True

    # Length: parse the whole segment between "которые идут" and rating/end.
    m = re.search(r"которые идут\s+(.+?)(?:\s+и\s+имеют\s+рейтинг|\s+с\s+рейтингом|[.;]|$)", text, flags=re.I)
    if m:
        mn, mx = minutes_from_length_phrase(m.group(1))
        if mn is not None:
            clean["movie_length_min_minutes"] = int(mn)
        if mx is not None:
            clean["movie_length_max_minutes"] = int(mx)

    return clean, full, warnings

def clean_countries(rec):
    clean, full, warnings = base_clean(rec)
    text = qtext(rec)
    add_if(clean, "region_ru", first_quoted_after(text, ["регионе", "региона", "континенте", "континента"]))
    # Generator uses "регион" for continent; keep human name as region_ru.
    add_if(clean, "official_language_ru", first_quoted_after(text, ["официальный язык", "официальных языков", "имеют официальный язык"]))
    add_if(clean, "currency_ru", first_quoted_after(text, ["валюта", "валютой"]))
    pop_min, pop_max = numeric_range_after(text, ["численность населения", "население"])
    if pop_min is not None: clean["population_min"] = int(pop_min)
    if pop_max is not None: clean["population_max"] = int(pop_max)
    if "population_min" not in clean:
        v = numeric_min_after(text, ["численность населения", "население"])
        if v is not None: clean["population_min"] = int(v)
    if "population_max" not in clean:
        v = numeric_max_after(text, ["численность населения", "население"])
        if v is not None: clean["population_max"] = int(v)
    tl = text.lower()
    if re.search(r"историческ|которых уже не существует|бывш", tl):
        clean["historical_only"] = True
        clean["exists_now"] = False
    elif re.search(r"современн|существуют сейчас|нынешн", tl):
        clean["historical_only"] = False
        clean["exists_now"] = True
    if re.search(r"не входят в европейский союз|не состоят в европейском союзе|не член.*европейск", tl):
        clean["not_member_of_ru"] = "Европейский союз"
    if re.search(r"европейск[а-я ]+союз", tl) and re.search(r"\bG7\b|больш[а-я ]+сем[её]р", text, flags=re.I):
        clean["member_of_ru"] = ["Европейский союз", "G7"]
    return clean, full, warnings

def clean_dishes(rec):
    clean, full, warnings = base_clean(rec)
    text = qtext(rec)
    qv = quoted_values(text)
    # include/exclude ingredients
    include, exclude = [], []
    for m in re.finditer(r"(?:где есть|содерж[аи][тщ]?.{0,10}|с ингредиентом|в составе есть)\s*[«\"]([^»\"]+)[»\"]", text, flags=re.I):
        include.append(clean_label(m.group(1)))
    for m in re.finditer(r"(?:нет|без|при этом нет|не содержит|не должны содержать)\s*[«\"]([^»\"]+)[»\"]", text, flags=re.I):
        exclude.append(clean_label(m.group(1)))
    if not include and qv:
        # first quoted value is usually positive ingredient/cuisine/category
        if re.search(r"где есть|содерж|ингредиент|в составе", text, flags=re.I):
            include.append(qv[0])
    add_if(clean, "include_ingredients_ru", include)
    add_if(clean, "exclude_ingredients_ru", exclude)
    add_if(clean, "country_ru", country_from_text(text))
    add_if(clean, "cuisine_ru", first_quoted_after(text, ["кухни", "кухня"]))
    add_if(clean, "category_ru", first_quoted_after(text, ["категории", "категория"]))
    return clean, full, warnings

def clean_books(rec):
    clean, full, warnings = base_clean(rec)
    text=qtext(rec)
    add_if(clean, "genre_ru", first_quoted_after(text, ["жанра", "жанре", "в жанре"]))
    add_if(clean, "country_ru", country_from_text(text))
    add_if(clean, "language_ru", first_quoted_after(text, ["языке", "язык", "написан"]))
    clean.update(year_constraints(text))
    if re.search(r"не\s+(?:входит|является частью).{0,50}(серии|цикла)", text, flags=re.I):
        clean["not_series"] = True
    if re.search(r"женщин[аы].{0,20}автор|автор[а-я]*\s*[-—]?\s*женщин", text, flags=re.I):
        clean["female_author"] = True
    if re.search(r"мужчин[аы].{0,20}автор|автор[а-я]*\s*[-—]?\s*мужчин", text, flags=re.I):
        clean["male_author"] = True
    add_if(clean, "setting_city_ru", first_quoted_after(text, ["действие происходит", "место действия", "в городе"]))
    v = numeric_max_after(text, ["опубликован", "издан", "публикац"])
    if v is not None: clean["publication_year_to"] = int(v)
    return clean, full, warnings

def clean_videogames(rec):
    clean, full, warnings = base_clean(rec)
    text=qtext(rec)
    add_if(clean, "genre_ru", first_quoted_after(text, ["жанра", "жанре", "в жанре"]))
    add_if(clean, "platform_ru", first_quoted_after(text, ["платформ", "для платформы"]))
    add_if(clean, "developer_ru", first_quoted_after(text, ["разработчик", "разработанные"]))
    add_if(clean, "publisher_ru", first_quoted_after(text, ["издател", "изданные"]))
    add_if(clean, "series_ru", first_quoted_after(text, ["серии", "франшизы"]))
    clean.update(year_constraints(text))
    return clean, full, warnings

def clean_music_albums(rec):
    clean, full, warnings = base_clean(rec)
    text=qtext(rec)
    add_if(clean, "genre_ru", first_quoted_after(text, ["жанра", "жанре", "в жанре"]))
    add_if(clean, "country_ru", country_from_text(text))
    add_if(clean, "performer_ru", first_quoted_after(text, ["исполнител", "артист", "групп"]))
    add_if(clean, "label_ru", first_quoted_after(text, ["лейбл", "звукозаписывающей компании"]))
    clean.update(year_constraints(text))
    return clean, full, warnings

def clean_software(rec):
    clean, full, warnings = base_clean(rec)
    text=qtext(rec)
    add_if(clean, "programming_language_ru", first_quoted_after(text, ["языке программирования", "написан", "язык программирования"]))
    add_if(clean, "operating_system_ru", first_quoted_after(text, ["операционной системы", "ОС", "платформы"]))
    add_if(clean, "license_ru", first_quoted_after(text, ["лиценз", "лицензией"]))
    clean.update(year_constraints(text))
    return clean, full, warnings

def clean_people_like(rec):
    clean, full, warnings = base_clean(rec)
    text=qtext(rec)
    add_if(clean, "occupation_ru", first_quoted_after(text, ["профессии", "род занятий", "являются", "занятие"]))
    add_if(clean, "citizenship_ru", country_from_text(text) or first_quoted_after(text, ["гражданство", "гражданином", "граждане"]))
    clean.update(year_constraints(text))
    m = re.search(r"родивш[а-я]+\s+в\s+(1[5-9]\d{2}|20\d{2})", text, flags=re.I)
    if m: clean["birth_year"] = int(m.group(1))
    if re.search(r"\bженщин", text, flags=re.I): clean["sex_ru"] = "женский пол"
    if re.search(r"\bмужчин", text, flags=re.I): clean["sex_ru"] = "мужской пол"
    add_if(clean, "award_ru", first_quoted_after(text, ["преми", "награду", "лауреат"]))
    # award range/year
    if "award_ru" in clean:
        yc = year_constraints(text)
        if "year_from" in yc: clean["award_year_from"] = yc["year_from"]
        if "year_to" in yc: clean["award_year_to"] = yc["year_to"]
    return clean, full, warnings

def clean_paintings(rec):
    clean, full, warnings = base_clean(rec)
    text=qtext(rec)
    add_if(clean, "genre_ru", first_quoted_after(text, ["жанра", "жанре"]))
    add_if(clean, "movement_ru", first_quoted_after(text, ["направления", "движения", "стиля"]))
    add_if(clean, "creator_ru", first_quoted_after(text, ["автора", "художника", "созданные"]))
    add_if(clean, "collection_ru", first_quoted_after(text, ["коллекции", "музее", "хранятся в"]))
    add_if(clean, "country_ru", country_from_text(text))
    add_if(clean, "creator_country_ru", first_quoted_after(text, ["страны автора", "автор из"]))
    add_if(clean, "collection_country_ru", first_quoted_after(text, ["страны коллекции", "музей в стране"]))
    add_if(clean, "depicts_ru", first_quoted_after(text, ["изображает", "изображающие", "depicts"]))
    add_if(clean, "material_ru", first_quoted_after(text, ["материал", "написанные на", "техника"]))
    clean.update(year_constraints(text))
    return clean, full, warnings

def clean_museums(rec):
    clean, full, warnings = base_clean(rec)
    text=qtext(rec)
    add_if(clean, "country_ru", country_from_text(text))
    add_if(clean, "type_ru", first_quoted_after(text, ["типа", "тип", "категории"]))
    add_if(clean, "city_ru", first_quoted_after(text, ["городе", "города"]))
    clean.update(year_constraints(text))
    return clean, full, warnings

def clean_spacecraft(rec):
    clean, full, warnings = base_clean(rec)
    text=qtext(rec)
    add_if(clean, "manufacturer_ru", first_quoted_after(text, ["производител", "изготовител"]))
    add_if(clean, "operator_ru", first_quoted_after(text, ["оператор", "эксплуатант"]))
    add_if(clean, "country_ru", country_from_text(text))
    add_if(clean, "program_ru", first_quoted_after(text, ["программ", "космической программе"]))
    clean.update(year_constraints(text))
    return clean, full, warnings

def clean_smartphones(rec):
    clean, full, warnings = base_clean(rec)
    text=qtext(rec)
    add_if(clean, "manufacturer_ru", first_quoted_after(text, ["производителя", "бренда", "компании"]))
    add_if(clean, "operating_system_ru", first_quoted_after(text, ["операционной системой", "ОС"]))
    clean.update(year_constraints(text))
    add_if(clean, "reference_model_ru", first_quoted_after(text, ["модели"]))
    if re.search(r"позже модели", text, flags=re.I): clean["release_relation"] = "after_reference_model"
    if re.search(r"раньше модели", text, flags=re.I): clean["release_relation"] = "before_reference_model"
    if re.search(r"позже модели.+но раньше модели", text, flags=re.I): clean["release_relation"] = "between_reference_models"
    return clean, full, warnings

def clean_universities(rec):
    clean, full, warnings = base_clean(rec)
    text=qtext(rec)
    add_if(clean, "country_ru", country_from_text(text))
    # macro region can be unquoted
    m = re.search(r"(?:регион[ае]?|макрорегион[ае]?)\s+[«\"]?([^,»\".;]+)[»\"]?", text, flags=re.I)
    if m: add_if(clean, "macro_region_ru", m.group(1))
    clean.update(year_constraints(text))
    return clean, full, warnings

def clean_airports(rec):
    clean, full, warnings = base_clean(rec)
    text=qtext(rec)
    add_if(clean, "country_ru", country_from_text(text))
    v = numeric_min_after(text, ["население", "пассажиропоток", "город"])
    if v is not None: clean["population_min"] = int(v)
    if re.search(r"\bIATA\b|код IATA|тр[её]хбуквен", text, flags=re.I):
        clean["require_iata_code"] = True
    return clean, full, warnings

def clean_cars(rec):
    clean, full, warnings = base_clean(rec)
    text=qtext(rec)
    # one or two countries
    qv = quoted_values(text)
    countries = []
    for m in re.finditer(r"(?:страны|странах|из)\s*[«\"]([^»\"]+)[»\"]", text, flags=re.I):
        countries.append(clean_label(m.group(1)))
    if not countries:
        c = country_from_text(text)
        if c: countries = [c]
    if len(countries) == 1:
        clean["country_ru"] = countries[0]
    elif len(countries) >= 2:
        clean["countries_ru"] = countries[:2]
    add_if(clean, "manufacturer_ru", first_quoted_after(text, ["производителя", "марки", "компании"]))
    clean.update(year_constraints(text))
    return clean, full, warnings

def clean_geo(rec):
    clean, full, warnings = base_clean(rec)
    text=qtext(rec)
    # object kind refinement
    for k_ru, k_val in [
        ("море", "sea"), ("озеро", "lake"), ("водопад", "waterfall"), ("пустын", "desert"),
        ("гора", "mountain"), ("вулкан", "volcano"), ("река", "river"), ("остров", "island"),
        ("город", "city"), ("село", "settlement"), ("лес", "forest"), ("каньон", "canyon"),
    ]:
        if k_ru in text.lower():
            clean["kind"] = k_val
            break
    add_if(clean, "country_ru", country_from_text(text))
    add_if(clean, "region_ru", first_quoted_after(text, ["регионе", "части", "округе", "области"]))
    add_if(clean, "located_in_ru", first_quoted_after(text, ["расположены в", "находятся в", "входит в"]))
    # generic numeric constraints
    for label, anchors in [
        ("elevation_min_m", ["высота", "высотой"]),
        ("depth_min_m", ["глубина", "глубиной"]),
        ("area_min_km2", ["площадь"]),
        ("length_min_km", ["длина", "длиной"]),
        ("population_min", ["население"]),
    ]:
        v = numeric_min_after(text, anchors)
        if v is not None: clean[label] = int(v)
    return clean, full, warnings

DOMAIN_CLEANERS = {
    "cinema": clean_cinema,
    "countries": clean_countries,
    "dishes": clean_dishes,
    "books": clean_books,
    "videogames": clean_videogames,
    "music_albums": clean_music_albums,
    "software": clean_software,
    "people": clean_people_like,
    "scientists": clean_people_like,
    "mathematicians": clean_people_like,
    "paintings": clean_paintings,
    "museums": clean_museums,
    "spacecraft": clean_spacecraft,
    "smartphones": clean_smartphones,
    "universities": clean_universities,
    "airports": clean_airports,
    "cars": clean_cars,
    "geo": clean_geo,
    "geo_ru": clean_geo,
}

# Safe fallback from old human-readable constraints ONLY when text parser missed a field.
OLD_HUMAN_KEY_MAP = {
    "genre_ru": "genre_ru",
    "country_ru": "country_ru",
    "lang_ru": "language_ru",
    "language_label_ru": "official_language_ru",  # countries only handled separately below
    "currency_label_ru": "currency_ru",
    "continent_label_ru": "region_ru",
    "performer_ru": "performer_ru",
    "label_ru": "label_ru",
    "developer_ru": "developer_ru",
    "publisher_ru": "publisher_ru",
    "platform_ru": "platform_ru",
    "series_ru": "series_ru",
    "manufacturer_ru": "manufacturer_ru",
    "operator_ru": "operator_ru",
    "maker_label_ru": "manufacturer_ru",
    "creator_ru": "creator_ru",
    "collection_ru": "collection_ru",
    "movement_ru": "movement_ru",
    "material_ru": "material_ru",
    "depicts_ru": "depicts_ru",
    "city_ru": "city_ru",
    "type_ru": "type_ru",
    "occupation_ru": "occupation_ru",
    "citizenship_ru": "citizenship_ru",
    "award_ru": "award_ru",
}

def add_safe_old_fallback(rec: Dict[str, Any], clean: Dict[str, Any], warnings: List[str]) -> None:
    old = rec.get("constraints") or {}
    if not isinstance(old, dict):
        return
    domain = rec.get("domain")
    # Do NOT use old country for cinema unless country is in text. That was the original bug.
    key_map = dict(OLD_HUMAN_KEY_MAP)
    if domain == "cinema" and "country_ru" not in clean:
        key_map.pop("country_ru", None)
    if domain == "countries":
        # map country generator labels correctly
        if "region_ru" not in clean and old.get("continent_label_ru"):
            clean["region_ru"] = old["continent_label_ru"]
        if "official_language_ru" not in clean and old.get("language_label_ru"):
            clean["official_language_ru"] = old["language_label_ru"]
        if "currency_ru" not in clean and old.get("currency_label_ru"):
            clean["currency_ru"] = old["currency_label_ru"]
        if old.get("country_status") == "historical":
            clean["historical_only"] = True; clean["exists_now"] = False
        elif old.get("country_status") == "current":
            clean.setdefault("historical_only", False); clean.setdefault("exists_now", True)
        for k_old,k_new in [("pop_min","population_min"),("pop_max","population_max")]:
            if k_new not in clean and old.get(k_old) is not None:
                clean[k_new]=old[k_old]
    clean_from_old_safe(old, clean, key_map)
    # generic old years
    for a,b in [("year_from","year_from"),("year_to","year_to"),("y1","year_from"),("y2","year_to"),
                ("pop_min","population_min"),("pop_max","population_max")]:
        if b not in clean and old.get(a) is not None:
            clean[b] = old[a]
    # boolean old constraints that are semantically safe only if matching text or not known to be polluted.
    if domain == "cinema":
        # keep old rating but normalize, because text may have source.
        if "not_franchise" not in clean and "not_series" not in clean and (old.get("not_franchise") is True or old.get("not_series") is True):
            if re.search(r"не .{0,60}(франшиз|серии|серия)", qtext(rec), flags=re.I):
                clean["not_franchise"] = True
        rating = old.get("rating")
        if isinstance(rating, (list, tuple)) and len(rating) >= 3:
            src, op, val = rating[0], rating[1], rating[2]
            src_norm = "kinopoisk" if str(src).lower() in {"kp","кинопоиск","kinopoisk"} else "imdb" if str(src).lower()=="imdb" else str(src).lower()
            if op in (">=", ">", "min", "от", "не ниже"):
                clean.setdefault(f"rating_{src_norm}_min", float(val))
            elif op in ("<=", "<", "max", "до", "не выше"):
                clean.setdefault(f"rating_{src_norm}_max", float(val))
    if domain == "dishes":
        if not clean.get("include_ingredients_ru") and old.get("ingredient_include_ru"):
            clean["include_ingredients_ru"] = old.get("ingredient_include_ru")
        if not clean.get("exclude_ingredients_ru") and old.get("ingredient_exclude_ru"):
            clean["exclude_ingredients_ru"] = old.get("ingredient_exclude_ru")

def normalize_clean_constraints(clean: Dict[str, Any]) -> Dict[str, Any]:
    # remove qids, metadata, nulls, and weird helper keys
    out = {}
    for k, v in clean.items():
        if k in FORBIDDEN_CLEAN_KEYS:
            continue
        if k.endswith("_qid") or k.endswith("_qids"):
            continue
        if v is None or v == "" or v == [] or v == {}:
            continue
        if isinstance(v, str) and re.fullmatch(r"Q\d+", v):
            continue
        if isinstance(v, tuple):
            v = list(v)
        out[k] = v
    # deterministic order: kind first, then common fields, then rest
    order = [
        "kind", "genre_ru", "genres_ru", "country_ru", "countries_ru", "region_ru",
        "official_language_ru", "currency_ru", "language_ru",
        "population_min", "population_max", "year_from", "year_to",
        "rating_kinopoisk_min", "rating_kinopoisk_max", "rating_imdb_min", "rating_imdb_max",
        "actor_ru", "director_ru", "female_director", "male_director",
        "movie_length_min_minutes", "movie_length_max_minutes",
        "not_franchise", "part_of_franchise", "not_series", "not_sequel", "not_remake",
        "include_ingredients_ru", "exclude_ingredients_ru", "cuisine_ru", "category_ru",
        "manufacturer_ru", "developer_ru", "publisher_ru", "platform_ru", "series_ru",
        "performer_ru", "label_ru", "programming_language_ru", "operating_system_ru", "license_ru",
        "occupation_ru", "citizenship_ru", "birth_year", "sex_ru", "award_ru",
        "historical_only", "exists_now", "not_member_of_ru", "member_of_ru",
    ]
    return {k: out[k] for k in order if k in out} | {k: out[k] for k in sorted(out) if k not in order}

def rebuild_record(rec: Dict[str, Any]) -> Tuple[Dict[str, Any], Dict[str, Any]]:
    old = rec.get("constraints", {})
    domain = rec.get("domain")
    cleaner = DOMAIN_CLEANERS.get(domain)
    if cleaner is None:
        clean, full, warnings = base_clean(rec)
        warnings.append(f"no_domain_cleaner:{domain}")
    else:
        clean, full, warnings = cleaner(rec)
    add_safe_old_fallback(rec, clean, warnings)
    clean = normalize_clean_constraints(clean)
    # audit: constraints empty for wikidata-like records
    if not clean and (rec.get("sparql_query") or rec.get("ask_validator_sparql")):
        warnings.append("empty_clean_constraints_after_rebuild")
    # audit common text conditions
    txt = qtext(rec).lower()
    if "населен" in txt and not any(k in clean for k in ("population_min","population_max")):
        warnings.append("population_mentioned_but_not_parsed")
    if "франшиз" in txt and not any(k in clean for k in ("not_franchise","part_of_franchise")):
        warnings.append("franchise_mentioned_but_not_parsed")
    if "рейтинг" in txt and not any(k.startswith("rating_") for k in clean):
        warnings.append("rating_mentioned_but_not_parsed")
    if "валюта" in txt and "currency_ru" not in clean:
        warnings.append("currency_mentioned_but_not_parsed")
    if "официальный язык" in txt and "official_language_ru" not in clean:
        warnings.append("official_language_mentioned_but_not_parsed")
    if ("где есть" in txt or "при этом нет" in txt or "не содержит" in txt) and domain == "dishes":
        if not clean.get("include_ingredients_ru") or not clean.get("exclude_ingredients_ru"):
            warnings.append("ingredient_include_or_exclude_maybe_missing")
    new = deepcopy(rec)
    new["constraints"] = clean
    new["full_constraints"] = {
        **full,
        "old_constraints_hash": stable_hash(old),
        "new_constraints_hash": stable_hash(clean),
        "text_evidence": {
            "query_text_ru": rec.get("query_text_ru"),
            "quoted_values": quoted_values(rec.get("query_text_ru") or ""),
        },
    }
    new["constraints_fix_meta"] = {
        "method": "domain_specific_constraints_v7_4_from_generation_templates",
        "warnings": sorted(set(warnings)),
    }
    return new, {
        "id": rec.get("id"),
        "domain": domain,
        "complexity": rec.get("complexity"),
        "template_id": rec.get("template_id"),
        "old_constraints": json.dumps(old, ensure_ascii=False, sort_keys=True),
        "new_constraints": json.dumps(clean, ensure_ascii=False, sort_keys=True),
        "warnings": ";".join(sorted(set(warnings))),
        "query_text_ru": rec.get("query_text_ru"),
    }

def load_jsonl(path: Path) -> List[Dict[str, Any]]:
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

def save_jsonl(records: List[Dict[str, Any]], path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        for r in records:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

def write_csv(rows: List[Dict[str, Any]], path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    if not rows:
        return
    with open(path, "w", encoding="utf-8", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        w.writeheader()
        w.writerows(rows)

def run_fix():
    print("cwd:", Path.cwd())
    print("input:", INPUT_JSONL.resolve())
    if not INPUT_JSONL.exists():
        # fallback recursive search
        matches = list(Path.cwd().rglob(FILENAME))
        if not matches:
            raise FileNotFoundError(f"Cannot find {FILENAME}")
        globals()["INPUT_JSONL"] = matches[0]
    records = load_jsonl(INPUT_JSONL)
    out, audit, diffs = [], [], []
    for rec in records:
        new, row = rebuild_record(rec)
        out.append(new)
        audit.append(row)
        if row["old_constraints"] != row["new_constraints"]:
            diffs.append(row)
    save_jsonl(out, OUTPUT_JSONL)
    write_csv(audit, AUDIT_CSV)
    write_csv(diffs, DIFF_CSV)
    print("records:", len(records))
    print("changed:", len(diffs))
    print("output:", OUTPUT_JSONL.resolve())
    print("diff:", DIFF_CSV.resolve())
    print("audit:", AUDIT_CSV.resolve())
    if OVERWRITE_INPUT:
        backup = INPUT_JSONL.with_suffix(INPUT_JSONL.suffix + ".before_constraints_v7.bak")
        backup.write_text(INPUT_JSONL.read_text(encoding="utf-8"), encoding="utf-8")
        INPUT_JSONL.write_text(OUTPUT_JSONL.read_text(encoding="utf-8"), encoding="utf-8")
        print("OVERWRITTEN input, backup:", backup.resolve())
    return out, audit, diffs


In [2]:
# Run the fix
records_fixed, audit_rows, diff_rows = run_fix()


cwd: /Users/matvey/Desktop/multihop benchmark
input: /Users/matvey/Desktop/multihop benchmark/merge_slices/merged_dataset/multihop_benchmark_merged_wikidata_format.jsonl
records: 1173
changed: 1173
output: /Users/matvey/Desktop/multihop benchmark/merge_slices/merged_dataset/multihop_benchmark_merged_wikidata_format.clean_constraints_v7_4_domain_specific.jsonl
diff: /Users/matvey/Desktop/multihop benchmark/merge_slices/merged_dataset/constraints_clean_v7_4_diff.csv
audit: /Users/matvey/Desktop/multihop benchmark/merge_slices/merged_dataset/constraints_clean_v7_4_audit.csv


In [3]:
# Sanity checks for known problematic patterns

def show_contains(substr: str, limit: int = 5):
    print("=" * 120)
    print("SEARCH:", substr)
    n = 0
    for r in records_fixed:
        if substr.lower() in (r.get("query_text_ru") or "").lower():
            print("\nID:", r.get("id"))
            print("DOMAIN:", r.get("domain"), "COMPLEXITY:", r.get("complexity"), "TEMPLATE:", r.get("template_id"))
            print("QUERY:", r.get("query_text_ru"))
            print("CONSTRAINTS:", json.dumps(r.get("constraints"), ensure_ascii=False, sort_keys=False))
            print("WARNINGS:", r.get("constraints_fix_meta", {}).get("warnings"))
            n += 1
            if n >= limit:
                break
    if n == 0:
        print("No rows found")

show_contains("рейтингом по Кинопоиск", 5)
show_contains("франшизы/серии", 5)
show_contains("численность населения примерно в диапазоне", 5)
show_contains("восточноафриканский шиллинг", 5)
show_contains("где есть «мука»", 5)
show_contains("при этом нет «говядина»", 5)


SEARCH: рейтингом по Кинопоиск

ID: cinema_l2_0040
DOMAIN: cinema COMPLEXITY: L2 TEMPLATE: None
QUERY: Назови 5 фильмов, жанра «боевик», в период 2021–2024 годов, с рейтингом по Кинопоиск не ниже 6.8.
CONSTRAINTS: {"kind": "film", "genre_ru": "боевик", "year_from": 2021, "year_to": 2024, "rating_kinopoisk_min": 6.8}
WARNINGS: []

ID: cinema_l2_0066
DOMAIN: cinema COMPLEXITY: L2 TEMPLATE: None
QUERY: Назови 5 фильмов, жанра «боевик», в период 2019–2025 годов, с рейтингом по Кинопоиск не выше 5.9.
CONSTRAINTS: {"kind": "film", "genre_ru": "боевик", "year_from": 2019, "year_to": 2025, "rating_kinopoisk_min": 5.9, "rating_kinopoisk_max": 5.9}
WARNINGS: []

ID: cinema_l2_0070
DOMAIN: cinema COMPLEXITY: L2 TEMPLATE: None
QUERY: Назови 5 фильмов, жанра «ужасы», в период 2020–2022 годов, с рейтингом по Кинопоиск не ниже 6.5.
CONSTRAINTS: {"kind": "film", "genre_ru": "ужасы", "year_from": 2020, "year_to": 2022, "rating_kinopoisk_min": 6.5}
WARNINGS: []

ID: cinema_l2_0072
DOMAIN: cinema COMPLEX

In [4]:
# Audit rows with warnings. These need manual inspection.
warned = [r for r in audit_rows if r.get("warnings")]
print("warning rows:", len(warned))
for r in warned[:30]:
    print("\n", r["id"], r["domain"], r["complexity"], r.get("template_id"))
    print("warnings:", r["warnings"])
    print("query:", r["query_text_ru"])
    print("new:", r["new_constraints"])


warning rows: 169

 dishes_l3_00158 dishes L3 dishes_L3_ingredient_NOT_ingredient
warnings: ingredient_include_or_exclude_maybe_missing
query: Назови 3 блюд, в составе которых есть «говядина», но при этом НЕТ ингредиента «свинина».
new: {"include_ingredients_ru": ["говядина"], "kind": "dish"}

 dishes_l5_00200 dishes L5 dishes_adv_two_ing_not_multihop
warnings: ingredient_include_or_exclude_maybe_missing
query: Подбери ровно 5 блюд, где одновременно есть «лук» и «рыба», при этом нет «рис», а страна происхождения находится в регионе «Азия».
new: {"country_ru": "происхождения находится в регионе «Азия", "exclude_ingredients_ru": ["рис"], "kind": "dish", "region_ru": "Азия"}

 dishes_l5_00202 dishes L5 dishes_adv_two_ing_not_multihop
warnings: ingredient_include_or_exclude_maybe_missing
query: Подбери ровно 5 блюд, где одновременно есть «сливочное масло» и «сыр», при этом нет «томат», а страна происхождения находится в регионе «Европа».
new: {"country_ru": "происхождения находится в регио

In [5]:
# Manual validation table in JSON only: two fields — query and clean constraints.
# Run this cell after `records_fixed, audit_rows, diff_rows = run_fix()`.

import json

MANUAL_VALIDATION_JSON = INPUT_JSONL.parent / "constraints_manual_validation.json"

manual_rows = []
for rec in records_fixed:
    # keep exactly two user-facing fields as requested
    manual_rows.append({
        "запрос": rec.get("query_text_ru", ""),
        "констрейнты": rec.get("constraints", {}),
    })

with MANUAL_VALIDATION_JSON.open("w", encoding="utf-8") as f:
    json.dump(manual_rows, f, ensure_ascii=False, indent=2)

print("manual validation JSON saved:", MANUAL_VALIDATION_JSON)
print("rows:", len(manual_rows))
print("preview:")
print(json.dumps(manual_rows[:3], ensure_ascii=False, indent=2))


manual validation JSON saved: merge_slices/merged_dataset/constraints_manual_validation.json
rows: 1173
preview:
[
  {
    "запрос": "Назови 5 фильмов, жанра «криминальный фильм», из страны: США, 2015 года.",
    "констрейнты": {
      "kind": "film",
      "genre_ru": "криминальный фильм",
      "country_ru": "США",
      "year_from": 2015,
      "year_to": 2015
    }
  },
  {
    "запрос": "Назови 5 фильмов, жанра «драма», в период 2018–2025 годов.",
    "констрейнты": {
      "kind": "film",
      "genre_ru": "драма",
      "year_from": 2018,
      "year_to": 2025
    }
  },
  {
    "запрос": "Назови 5 фильмов, жанра «криминальный фильм», 2019 года.",
    "констрейнты": {
      "kind": "film",
      "genre_ru": "криминальный фильм",
      "year_from": 2019,
      "year_to": 2019
    }
  }
]


In [6]:
# Optional overwrite after manual inspection.
# Keep this disabled until you inspect constraints_clean_v7_audit.csv and constraints_clean_v7_diff.csv.

if OVERWRITE_INPUT:
    print("Already overwritten by run_fix().")
else:
    print("Safe mode: original input was NOT overwritten.")
    print("Fixed file:", OUTPUT_JSONL)
    print("To overwrite manually after inspection, either set OVERWRITE_INPUT=True and rerun, or run:")
    print(f"cp {OUTPUT_JSONL} {INPUT_JSONL}")


Safe mode: original input was NOT overwritten.
Fixed file: merge_slices/merged_dataset/multihop_benchmark_merged_wikidata_format.clean_constraints_v7_4_domain_specific.jsonl
To overwrite manually after inspection, either set OVERWRITE_INPUT=True and rerun, or run:
cp merge_slices/merged_dataset/multihop_benchmark_merged_wikidata_format.clean_constraints_v7_4_domain_specific.jsonl merge_slices/merged_dataset/multihop_benchmark_merged_wikidata_format.jsonl
